# Assignment: Adult Census Income Classification
**Dataset:** Adult Census Income Dataset (Kaggle)  
**Target Variable:** Income status (>50K or <=50K)

**Name:** Gaurav Gour

**Reg No:** 23Bsa10096


In [1]:
import os
import numpy as np
import pandas as pd
import kagglehub

# download the dataset using kagglehub
path = kagglehub.dataset_download("priyamchoksi/adult-census-income-dataset")
print("Downloaded to:", path)

# extract the exact csv file location
files = os.listdir(path)
csv_file = [f for f in files if f.endswith('.csv')][0]
dataset_path = os.path.join(path, csv_file)
print("Using dataset file:", dataset_path)

100%|██████████| 450k/450k [00:00<00:00, 635kB/s]

Extracting files...
Downloaded to: /root/.cache/kagglehub/datasets/priyamchoksi/adult-census-income-dataset/versions/1
Using dataset file: /root/.cache/kagglehub/datasets/priyamchoksi/adult-census-income-dataset/versions/1/adult.csv


## Task 1: Dataset Understanding
Loading the dataset, checking the dimensions, structural info, and investigating the target variable's class distribution.

In [2]:
# load data and inspect structures
df = pd.read_csv(dataset_path)

print("Data dimensions:", df.shape)
print("\n--- Summary Info ---")
df.info()

print("\n--- Class Distribution ---")
print(df['income'].value_counts())

df.head()

Data dimensions: (32561, 15)

--- Summary Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education.num   32561 non-null  int64 
 5   marital.status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital.gain    32561 non-null  int64 
 11  capital.loss    32561 non-null  int64 
 12  hours.per.week  32561 non-null  int64 
 13  native.country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB

--- Class Distribution ---
income
<=50K  

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


## Task 2: Data Cleaning
Handling format irregularities by stripping whitespaces, identifying hidden missing values (`?`), imputing missing values with the column modes, and removing duplicate records.

In [3]:
# strip trailing spaces from text columns
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip()

# flag '?' marks as NaN
df.replace('?', np.nan, inplace=True)
print("Missing values found:\n", df.isnull().sum())

# fill missing rows using the column mode
for col in ['workclass', 'occupation', 'native.country']:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

# check and drop any duplicate records
print("\nDuplicate row count:", df.duplicated().sum())
df.drop_duplicates(inplace=True)

print("Null checks remaining:", df.isnull().sum().sum())

Missing values found:
 age                  0
workclass         1836
fnlwgt               0
education            0
education.num        0
marital.status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week       0
native.country     583
income               0
dtype: int64

Duplicate row count: 24
Null checks remaining: 0


## Task 3: Feature Engineering
Encoding the target variable into a binary format, converting categorical features into dummy variables via one-hot encoding, and applying standard scaling to the feature set.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# map income field to 0 and 1
df['income'] = df['income'].apply(lambda x: 1 if '>50K' in str(x) else 0)

X = df.drop(columns=['income'])
y = df['income']

# dummy encoding for categories
X = pd.get_dummies(X, drop_first=True)

# stratified train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train shape:", X_train_scaled.shape)
print("X_test shape:", X_test_scaled.shape)

X_train shape: (26029, 97)
X_test shape: (6508, 97)


## Task 4: Model Building
Initializing and training the five classification algorithms on the scaled training data.

In [5]:
import warnings
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(probability=True, random_state=42, max_iter=5000)
}

results_list = []

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_scaled, y_train)

    predictions = model.predict(X_test_scaled)
    probabilities = model.predict_proba(X_test_scaled)[:, 1]

    results_list.append({
        "Algorithm": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1 Score": f1_score(y_test, predictions),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    })

Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Training KNN...
Training SVM...


## Task 5: Performance Evaluation
Compiling the calculated metrics into the final evaluation matrix for comparison.

In [6]:
# generate the final evaluation summary table
final_report = pd.DataFrame(results_list).set_index("Algorithm")
final_report

,Accuracy,Precision,Recall,F1 Score,ROC-AUC
Algorithm,,,,,
Logistic Regression,0.850492,0.735178,0.593112,0.656548,0.898820
Decision Tree,0.806546,0.596623,0.608418,0.602463,0.738926
Random Forest,0.848341,0.720913,0.604592,0.657648,0.893274
KNN,0.820836,0.649331,0.557398,0.599863,0.826029
SVM,0.841887,0.725901,0.552296,0.627309,0.884069
